# 第8章：LLM工程实践 - 交互式学习

本notebook提供LLM工程实践的交互式学习体验。

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import numpy as np
import matplotlib.pyplot as plt

if not Path("notebooks/bootstrap.py").exists():
    root = Path("/content/signal-to-intelligence")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/lynnyulinlin-debug/signal-to-intelligence.git", str(root)],
            check=True,
        )
    os.chdir(root)

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

from notebooks.bootstrap import load_code_module

engineering = load_code_module("code/ch08_llm_engineering/llm_engineering_demo.py")

%matplotlib inline

selector = engineering.ModelSelector()
analyzer = engineering.CostAnalyzer(selector)
models = ['gpt-4', 'gpt-3.5-turbo', 'claude-3-sonnet', 'llama-2-70b']
estimates = [
    analyzer.estimate_cost(
        model,
        daily_requests=10_000,
        avg_input_tokens=800,
        avg_output_tokens=300,
    )
    for model in models
]

fig, ax = plt.subplots(figsize=(10, 6))
monthly_costs = [estimate.monthly_cost for estimate in estimates]
bars = ax.barh(models, monthly_costs, color='steelblue', alpha=0.75)
ax.set_xlabel('Monthly Cost (USD)')
ax.set_title('LLM应用月成本估算')
ax.grid(True, alpha=0.3, axis='x')

for bar, cost in zip(bars, monthly_costs):
    ax.text(cost + max(monthly_costs) * 0.01, bar.get_y() + bar.get_height() / 2, f'${cost:.2f}', va='center')

plt.tight_layout()
plt.show()

print("\n成本优化策略:")
for key, strategy in analyzer.optimization_strategies().items():
    print(f"  {key}: {strategy}")
